# Notebook 01 — Load and Explore the Dataset

**Goal:** Load our football match data into Python and understand what we have.

**This notebook covers pipeline steps:**
1. Load dataset
2. Explore dataset (first look)

**How to run this notebook:**
1. Open a terminal in the project folder
2. Run: `jupyter notebook`
3. Open this file in the browser
4. Run cells **top to bottom** (Shift + Enter on each cell)

# Step 1 — Import Libraries

Before we touch the data, we import the tools we need.

- **pandas** — loads CSV files and works with tables (like Excel in Python)
- **Path** — helps us build file paths that work on Windows and Mac

We are **not** using scikit-learn or matplotlib yet — those come in later steps.

In [1]:
# Import pandas — our main tool for reading and exploring tabular data
import pandas as pd

# Path helps us locate files safely across operating systems
from pathlib import Path

# Confirm pandas loaded correctly (shows version number)
print("Pandas version:", pd.__version__)

Pandas version: 3.0.3


# Step 2 — Define the Path to Our Data

We always load data from `data/raw/` — the **original, untouched** copy.

Using `Path` means our code works even if you move the project folder.

In [3]:
# Build path: go up one folder from notebooks/ to project root, then into data/raw/
PROJECT_ROOT = Path("..")  # notebooks/ -> worldcup_predictor/
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "results.csv"

# Check the file exists before we try to read it (good debugging habit)
print("Looking for file at:", DATA_PATH.resolve())
print("File exists:", DATA_PATH.exists())

Looking for file at: C:\Users\HP-15\Desktop\worldcup_predictor\data\raw\results.csv
File exists: True


# Step 3 — Load the Dataset

`pd.read_csv()` reads a CSV file and stores it in a **DataFrame**.

**DataFrame** = a table in Python (rows = matches, columns = date, teams, scores, etc.)

In [4]:
# read_csv() opens the CSV and loads every row into a DataFrame
# We store it in a variable called 'df' (short for DataFrame — industry standard name)
df = pd.read_csv(DATA_PATH)

# Show how many rows and columns we loaded
print("Data loaded successfully!")
print("Shape (rows, columns):", df.shape)

Data loaded successfully!
Shape (rows, columns): (49477, 9)


# Step 4 — First Look at the Data

`.head()` shows the **first 5 rows** — like peeking at the top of a spreadsheet.

This is always the first thing data scientists do after loading data.

# Step 5 — Column Names and Data Types

- **`.columns`** — lists all column names (our potential features)
- **`.info()`** — shows each column's data type and how many non-empty values exist

**Data type examples:**
- `object` = text (team names, tournament names)
- `int64` = whole numbers (scores)
- `bool` = True/False (neutral venue flag)

In [5]:
# head() returns the first 5 rows by default
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [ ]:
# List all column names
print("Column names:")
print(list(df.columns))

print("\n" + "=" * 50 + "\n")

# info() gives a summary: column names, types, non-null counts
df.info()

# Step 6 — Basic Statistics

`.describe()` shows quick math stats for **number columns** only.

Useful to spot unusual values (e.g., negative scores would be a data error).

In [6]:
# describe() shows count, mean, min, max for numeric columns
df.describe()

,home_score,away_score
count,49433.000000,49433.000000
mean,1.757490,1.181822
std,1.774299,1.401856
min,0.000000,0.000000
25%,1.000000,0.000000
50%,1.000000,1.000000
75%,2.000000,2.000000
max,31.000000,21.000000


# Step 7 — What Tournaments Are in the Data?

Our project focuses on **World Cup** matches.

Let's see how many unique tournament names exist and sample some World Cup rows.

In [ ]:
# value_counts() counts how many matches per tournament (top 10 shown)
print("Top 10 tournaments by number of matches:")
print(df["tournament"].value_counts().head(10))

In [7]:
# Filter rows where tournament name contains 'World Cup'
# .str.contains() searches inside text columns
world_cup_mask = df["tournament"].str.contains("World Cup", case=False, na=False)
world_cup_df = df[world_cup_mask]

print("Total matches in full dataset:", len(df))
print("World Cup matches only:", len(world_cup_df))
print("\nSample World Cup matches:")
world_cup_df.head()

Total matches in full dataset: 49477
World Cup matches only: 9868

Sample World Cup matches:


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
1490,1930-07-13,Belgium,United States,0.0,3.0,FIFA World Cup,Montevideo,Uruguay,True
1491,1930-07-13,France,Mexico,4.0,1.0,FIFA World Cup,Montevideo,Uruguay,True
1492,1930-07-14,Brazil,Yugoslavia,1.0,2.0,FIFA World Cup,Montevideo,Uruguay,True
1493,1930-07-14,Peru,Romania,1.0,3.0,FIFA World Cup,Montevideo,Uruguay,True
1494,1930-07-15,Argentina,France,1.0,0.0,FIFA World Cup,Montevideo,Uruguay,True


# Step 8 — Identify Features and Target

**Features (inputs)** — information about the match we can use to predict:
- `home_team`, `away_team`, `tournament`, `neutral`, etc.

**Target (output)** — what we want to predict:
- Home team result: **Win**, **Draw**, or **Loss**

The raw file has `home_score` and `away_score`. We will **create** the target column in a later notebook (feature engineering step).

⚠️ **Important:** We must NOT use `home_score` or `away_score` as features — that would be cheating (we don't know the score before the match ends).

In [8]:
# Preview: create target column temporarily just to understand the concept
# We will do this properly in a later step

def get_home_result(row):
    """Returns Win, Draw, or Loss for the home team based on scores."""
    if row["home_score"] > row["away_score"]:
        return "Win"
    elif row["home_score"] == row["away_score"]:
        return "Draw"
    else:
        return "Loss"

# Apply function to World Cup sample only (for learning — not saved yet)
preview = world_cup_df.head(10).copy()
preview["home_result"] = preview.apply(get_home_result, axis=1)

preview[["home_team", "away_team", "home_score", "away_score", "home_result"]]

,home_team,away_team,home_score,away_score,home_result
1490,Belgium,United States,0.0,3.0,Loss
1491,France,Mexico,4.0,1.0,Win
1492,Brazil,Yugoslavia,1.0,2.0,Loss
1493,Peru,Romania,1.0,3.0,Loss
1494,Argentina,France,1.0,0.0,Win
1495,Chile,Mexico,3.0,0.0,Win
1496,Bolivia,Yugoslavia,0.0,4.0,Loss
1497,Paraguay,United States,0.0,3.0,Loss
1499,Uruguay,Peru,1.0,0.0,Win
1500,Argentina,Mexico,6.0,3.0,Win


# Summary — What We Learned in This Notebook

| Item | Value |
|------|-------|
| File loaded | `data/raw/results.csv` |
| Total matches | See `df.shape` output above |
| World Cup matches | See `len(world_cup_df)` above |
| Features | Team names, tournament, neutral flag, date, etc. |
| Target (to create) | `home_result` → Win / Draw / Loss |

**Next notebook:** Clean data and filter to World Cup matches only.